In [ ]:
import os
import matplotlib.pyplot as plt

from lsmcpp.planners import *
from lsmcpp.local_search import *
from lsmcpp.turn_minimization.TMSTCStar import TMSTCStar_planner

### Load the selected MCPP instance

In [ ]:
name = "floor_large"

# Unzip data files
if not os.path.exists(f"data/gridmaps/{name}.map"):
    import subprocess
    subprocess.call(["unzip", "-o", "data/data.zip", "-d", "data"])

mcpp = MCPP.read_instance(os.path.join("data", "instances", f"{name}.mcpp"))

fig, ax = plt.subplots(figsize=(8, 8), layout="constrained")
mcpp.draw(ax)

### Run LS-MCPP with MFC as initial solution and TMSTC*

In [ ]:
init_sol = MFC_planner(mcpp)
ls_mcpp_planner = LocalSearchMCPP(mcpp, init_sol, PrioType.CompositeHeur, PoolType.VertexEdgewise)
M = 2000
sol_opt, rt = ls_mcpp_planner.run(
    M=M,
    S=int(M // 20),
    alpha=np.exp(np.log(0.2) / M),
    gamma=0.01,
    sample_type=SampleType.RouletteWheel
)

tmstcstar_sol = TMSTCStar_planner(mcpp)


In [ ]:
colors = plt.get_cmap("tab20")(np.linspace(0, 1, mcpp.k))
fig, axs = plt.subplots(1, 2, figsize=(16, 8), layout="constrained")
for ax, solution, title in zip(axs, [sol_opt, tmstcstar_sol], ["LS-MCPP", "TMSTC*"]):
    mcpp.draw_obstacles(ax)
    for i, pi in enumerate(solution.Pi):
        xs, ys = zip(*pi)
        ax.plot(xs, ys, color=colors[i], linewidth=1.5, alpha=0.6)
        r = mcpp.legacy_vertex(mcpp.R[i])
        ax.plot(r[0], r[1], "*k", ms=8, mfc=colors[i], zorder=3)
    ax.set_title(title)
    ax.axis("equal")